# ODI to Databricks Migration

**Source File:** `TARGET_FILE.txt`
**Conversion Timestamp:** 2024-07-30 12:00:00
**Description:** This notebook performs a direct insert of location data from a source table into a target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT ${DATASOURCE_NUM_ID} AS datasource_num_id;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT ${ETL_PROC_WID} AS etl_proc_wid;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
print("ETL Parameters:")
display(spark.sql("SELECT * FROM v_etl_job_type"))
display(spark.sql("SELECT * FROM v_datasource_num_id"))
display(spark.sql("SELECT * FROM v_etl_proc_wid"))
display(spark.sql("SELECT * FROM v_odi_sess_no"))

# Load Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Initial setup/parameter handling (no explicit SQL in source)
-- SCEN_TASK_NO in {20}: Pre-load operations (no explicit SQL in source)
-- SCEN_TASK_NO in {30}: Pre-load operations (no explicit SQL in source)
INSERT
  INTO workspace.hr.trg_loc
  (
    location_id ,
    street_address ,
    postal_code ,
    city ,
    state_province ,
    country_id
  )
SELECT
  locations.location_id ,
  locations.street_address ,
  locations.postal_code ,
  locations.city ,
  locations.state_province ,
  locations.country_id
FROM
  workspace.hr.locations AS locations;

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were created in this process, so no explicit drops needed.

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_target
FROM workspace.hr.trg_loc;

# Conversion Notes

**Automatic Conversions:**
- Oracle hints `/*+ APPEND PARALLEL */` removed.
- Schema and table names converted to lowercase and prefixed with `workspace.` (e.g., `HR.TRG_LOC` -> `workspace.hr.trg_loc`).
- `SCEN_TASK_NO` placeholders were identified and noted as comments in the relevant code cell.

**Manual Actions Required (if applicable for a complete ETL flow):**
- **Target Table DDL:** Ensure `workspace.hr.trg_loc` table exists with appropriate Spark SQL data types (e.g., `VARCHAR2` -> `STRING`, `NUMBER` -> `BIGINT`/`DECIMAL`, `DATE` -> `TIMESTAMP`). The DDL was not part of the source input.
- **Error Handling:** No `E$_` error table processing was present in the source, so no error handling logic was generated.
- **Incremental Logic:** The original `INSERT` with `APPEND` suggests an append-only load. If `TRG_LOC` should be upserted or fully replaced, a `MERGE INTO` or `INSERT OVERWRITE` pattern might be more appropriate, but was not inferred from the minimal source. This conversion strictly follows the provided `INSERT` statement.